# Week 13 Walkthrough — Building a Robot That Senses

**Topic:** Grid worlds, robot classes, sensors, the sense-think-act loop

Everything from the semester, pointed at one thing: a virtual robot in a grid world. This week we build the world, put a robot in it, and give it sensors. Next week it starts making its own decisions.

Run the cells in order. Each step adds one idea to the program, and the last
section pulls the whole thing together. Change things and re-run — that is the
whole point of a notebook.

## Step 1 — The world is a grid


A list of lists — `0` is open floor, `1` is an obstacle. Row and column indexes
are all the coordinates we need.

In [ ]:
GRID = [
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 1, 0, 1, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0],
]

for row in GRID:
    print(" ".join("#" if cell else "." for cell in row))

## Step 2 — A GridWorld class


Wrap the grid so the rules about it — what's in bounds, what's blocked — live
with the data.

In [ ]:
class GridWorld:
    def __init__(self, grid, goal):
        self.grid = grid
        self.rows = len(grid)
        self.cols = len(grid[0])
        self.goal = goal

    def in_bounds(self, row, col):
        return 0 <= row < self.rows and 0 <= col < self.cols

    def is_open(self, row, col):
        return self.in_bounds(row, col) and self.grid[row][col] == 0

    def is_goal(self, row, col):
        return (row, col) == self.goal


world = GridWorld(GRID, goal=(5, 5))
print(world.is_open(0, 0), world.is_open(1, 1), world.is_open(99, 0))
print(world.is_goal(5, 5))

## Step 3 — A robot that knows where it is


Position, heading, and a log. That's the whole state.

In [ ]:
class Robot:
    HEADINGS = ["north", "east", "south", "west"]
    MOVES = {"north": (-1, 0), "east": (0, 1), "south": (1, 0), "west": (0, -1)}

    def __init__(self, world, row=0, col=0, heading="east"):
        self.world = world
        self.row = row
        self.col = col
        self.heading = heading
        self.log = []

    def __str__(self):
        return f"Robot at ({self.row}, {self.col}) facing {self.heading}"


robot = Robot(world)
print(robot)

## Step 4 — Acting: turn and move


Movement checks the world first. A robot that can walk through walls teaches you
nothing.

In [ ]:
class Robot(Robot):                     # extend the class we just made
    def turn_right(self):
        i = self.HEADINGS.index(self.heading)
        self.heading = self.HEADINGS[(i + 1) % 4]
        self.log.append(f"turn right -> {self.heading}")

    def turn_left(self):
        i = self.HEADINGS.index(self.heading)
        self.heading = self.HEADINGS[(i - 1) % 4]
        self.log.append(f"turn left -> {self.heading}")

    def ahead(self):
        """The square directly in front, as (row, col)."""
        dr, dc = self.MOVES[self.heading]
        return self.row + dr, self.col + dc

    def move_forward(self):
        row, col = self.ahead()
        if not self.world.is_open(row, col):
            self.log.append("blocked")
            return False
        self.row, self.col = row, col
        self.log.append(f"move -> ({row}, {col})")
        return True


robot = Robot(world)
print(robot.move_forward(), robot)
robot.turn_right()
print(robot.move_forward(), robot)

## Step 5 — Sensing: what can the robot detect?


Real robots don't know the map — they know what their sensors report right now.
Keep that honest and the simulation teaches the right lessons.

In [ ]:
class Robot(Robot):
    def obstacle_ahead(self):
        return not self.world.is_open(*self.ahead())

    def at_goal(self):
        return self.world.is_goal(self.row, self.col)

    def distance_to_wall(self):
        """How many open squares ahead before something blocks us."""
        dr, dc = self.MOVES[self.heading]
        steps, row, col = 0, self.row, self.col
        while self.world.is_open(row + dr, col + dc):
            row, col = row + dr, col + dc
            steps += 1
        return steps


robot = Robot(world)
print("obstacle ahead?", robot.obstacle_ahead())
print("clear run of  ", robot.distance_to_wall())
print("at goal?      ", robot.at_goal())

## Step 6 — Draw the world with the robot in it


Being able to *see* the state is the difference between debugging in minutes and
debugging in hours.

In [ ]:
ARROWS = {"north": "^", "east": ">", "south": "v", "west": "<"}

def render(world, robot):
    lines = []
    for r in range(world.rows):
        cells = []
        for c in range(world.cols):
            if (r, c) == (robot.row, robot.col):
                cells.append(ARROWS[robot.heading])
            elif world.is_goal(r, c):
                cells.append("G")
            elif world.grid[r][c]:
                cells.append("#")
            else:
                cells.append(".")
        lines.append(" ".join(cells))
    return "\n".join(lines)


robot = Robot(world)
print(render(world, robot))

## Step 7 — Sense, think, act — one step at a time


The loop at the heart of robotics. Sense the world, decide, do one thing, repeat.
Here the "thinking" is deliberately dumb: go straight if you can, otherwise turn.

In [ ]:
robot = Robot(world)

for tick in range(6):
    if robot.obstacle_ahead():          # SENSE + THINK
        robot.turn_right()              # ACT
    else:
        robot.move_forward()

print(render(world, robot))
print()
print(robot)
for entry in robot.log:
    print(" ", entry)

---

## The finished program

Everything above, in one place. This is the version worth keeping.


World, robot, sensors, renderer, and a short autonomous run — in one cell you can
copy straight into your capstone.

In [ ]:
# Week 13 - Grid world and a sensing robot

class GridWorld:
    def __init__(self, grid, goal):
        self.grid = grid
        self.rows, self.cols = len(grid), len(grid[0])
        self.goal = goal

    def in_bounds(self, r, c):
        return 0 <= r < self.rows and 0 <= c < self.cols

    def is_open(self, r, c):
        return self.in_bounds(r, c) and self.grid[r][c] == 0

    def is_goal(self, r, c):
        return (r, c) == self.goal


class Robot:
    HEADINGS = ["north", "east", "south", "west"]
    MOVES = {"north": (-1, 0), "east": (0, 1), "south": (1, 0), "west": (0, -1)}
    ARROWS = {"north": "^", "east": ">", "south": "v", "west": "<"}

    def __init__(self, world, row=0, col=0, heading="east"):
        self.world, self.row, self.col, self.heading = world, row, col, heading
        self.log = []

    # --- sensing
    def ahead(self):
        dr, dc = self.MOVES[self.heading]
        return self.row + dr, self.col + dc

    def obstacle_ahead(self):
        return not self.world.is_open(*self.ahead())

    def at_goal(self):
        return self.world.is_goal(self.row, self.col)

    # --- acting
    def turn_right(self):
        self.heading = self.HEADINGS[(self.HEADINGS.index(self.heading) + 1) % 4]
        self.log.append(f"turn -> {self.heading}")

    def move_forward(self):
        r, c = self.ahead()
        if not self.world.is_open(r, c):
            self.log.append("blocked")
            return False
        self.row, self.col = r, c
        self.log.append(f"move -> ({r}, {c})")
        return True

    def render(self):
        out = []
        for r in range(self.world.rows):
            row = []
            for c in range(self.world.cols):
                if (r, c) == (self.row, self.col):
                    row.append(self.ARROWS[self.heading])
                elif self.world.is_goal(r, c):
                    row.append("G")
                elif self.world.grid[r][c]:
                    row.append("#")
                else:
                    row.append(".")
            out.append(" ".join(row))
        return "\n".join(out)

    def __str__(self):
        return f"Robot at ({self.row}, {self.col}) facing {self.heading}"


GRID = [
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 1, 0, 1, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0],
]

world = GridWorld(GRID, goal=(5, 5))
robot = Robot(world)

print("START")
print(robot.render())

for tick in range(12):
    if robot.at_goal():
        break
    if robot.obstacle_ahead():
        robot.turn_right()
    else:
        robot.move_forward()

print("\nAFTER 12 TICKS")
print(robot.render())
print(f"\n{robot} - reached goal: {robot.at_goal()}")

---

## Try it yourself

Use the empty cells below. There is no grade attached — this is where the
learning actually happens.

**1.** Add `turn_left()` and confirm four left turns bring the heading back to where it started.

**2.** Add a `battery` attribute that starts at 20 and drops by 1 per move. Stop the loop when it hits zero.

**3.** Change the grid — add a wall, move the goal — and re-run. The robot code shouldn't need a single edit.

**4.** Print `robot.render()` inside the loop to watch it move tick by tick.

In [ ]:
# Try it yourself 1

In [ ]:
# Try it yourself 2

In [ ]:
# Try it yourself 3